In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
from google.colab import files
upload = files.upload()

Saving credit_risk_synthetic.csv to credit_risk_synthetic.csv


In [3]:

df = pd.read_csv('credit_risk_synthetic.csv')
df.head()

,age,income,loan_amount,loan_term_months,credit_score,employment_years,num_prior_defaults,home_ownership,purpose,default
0,41,165766,575961,40,647,7,0,rent,auto,1
1,28,140095,768566,35,719,10,0,mortgage,education,1
2,46,272081,581996,52,717,0,0,rent,personal,1
3,47,250223,1171649,47,584,5,0,mortgage,personal,1
4,18,51425,928936,6,636,9,1,rent,education,1


In [4]:
# Preprocessing
df = df.drop(columns=['purpose'])

le = LabelEncoder()
df['home_ownership'] = le.fit_transform(df['home_ownership'])

X = df.drop('default', axis=1)
y = df['default']

In [5]:
z_scores = np.abs((X - X.mean()) / X.std())
df_cleaned = df[(z_scores < 3).all(axis=1)]

X = df_cleaned.drop('default', axis=1)
y = df_cleaned['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1902 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   age                 1902 non-null   int64
 1   income              1902 non-null   int64
 2   loan_amount         1902 non-null   int64
 3   loan_term_months    1902 non-null   int64
 4   credit_score        1902 non-null   int64
 5   employment_years    1902 non-null   int64
 6   num_prior_defaults  1902 non-null   int64
 7   home_ownership      1902 non-null   int64
 8   default             1902 non-null   int64
dtypes: int64(9)
memory usage: 148.6 KB
None


In [6]:
print(df_cleaned.head())

   age  income  loan_amount  loan_term_months  credit_score  employment_years  \
0   41  165766       575961                40           647                 7   
1   28  140095       768566                35           719                10   
2   46  272081       581996                52           717                 0   
3   47  250223      1171649                47           584                 5   
4   18   51425       928936                 6           636                 9   

   num_prior_defaults  home_ownership  default  
0                   0               2        1  
1                   0               0        1  
2                   0               2        1  
3                   0               0        1  
4                   1               2        1  


In [7]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train)
lr_acc = accuracy_score(y_test, lr_model.predict(X_test_scaled))

In [8]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(
    estimator=SVC(probability=True),
    param_grid=param_grid,
    cv=5,
    verbose=2
)

print("\n--- Starting Grid Search ---")
grid_search.fit(X_train_scaled, y_train)

svm_model = grid_search.best_estimator_
y_pred_best = svm_model.predict(X_test_scaled)


--- Starting Grid Search ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.4s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.5s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.4s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.3s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.5s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.2s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.3s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.3s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.1s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.3s
[CV] END ......................C=0.1, gamma=auto, kernel=rbf; total time=   0.4s
[CV] END .........

In [9]:
print("\nParameters Found:", grid_search.best_params_)

print("\n--- Optimized SVM Performance ---")
print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")

print("\nClassification Report:\n", classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_best))


Parameters Found: {'C': 100, 'gamma': 'scale', 'kernel': 'linear'}

--- Optimized SVM Performance ---
Test Set Accuracy: 0.9843

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.94      0.95        67
           1       0.99      0.99      0.99       314

    accuracy                           0.98       381
   macro avg       0.98      0.97      0.97       381
weighted avg       0.98      0.98      0.98       381


Confusion Matrix:
 [[ 63   4]
 [  2 312]]
